# API Routes - Effects
Flask routes for EM effect management operations.

In [ ]:
# Effects API routes

def register_effects_routes(app, get_h5_file, get_effects_grid, set_effects_grid, create_sequencer_effect, hruuid, datetime_module):
    """Register effect-related API routes"""
    from flask import jsonify, request
    import random
    
    @app.route('/api/effects', methods=['GET'])
    def get_effects():
        """Get current effects grid state"""
        try:
            grid = get_effects_grid()
            effect_types_grid = []
            with get_h5_file() as f:
                effects_props_group = f.require_group("sequencer_effects_properties")
                for row in grid:
                    effect_types_row = []
                    for uuid in row:
                        if uuid:
                            if uuid in effects_props_group.keys():
                                sequencer_effect = effects_props_group[uuid]
                                effect_type = sequencer_effect.attrs.get("effect_type", "")
                                effect_types_row.append(effect_type)
                            else:
                                effect_types_row.append("")
                        else:
                            effect_types_row.append("")
                    effect_types_grid.append(effect_types_row)
            return jsonify({"effects": effect_types_grid}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/effects', methods=['PUT'])
    def update_effects():
        """Update effects grid position"""
        try:
            data = request.get_json()
            row = data.get('row')
            col = data.get('col')
            effect = data.get('effect', '')
            properties = data.get('properties', {})
            
            if row is None or col is None:
                return jsonify({"error": "row and col are required"}), 400
            
            if row < 0 or row >= 2 or col < 0 or col >= 12:
                return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
            
            valid_effects = ['AC', 'DC', 'AMF', 'CMF', '']
            if effect not in valid_effects:
                return jsonify({"error": f"effect must be one of {valid_effects}"}), 400
            
            grid = get_effects_grid()
            
            with get_h5_file() as f:
                effects_props_group = f.require_group("sequencer_effects_properties")
                
                if not effect:
                    old_uuid = grid[row][col]
                    if old_uuid and old_uuid in effects_props_group.keys():
                        del effects_props_group[old_uuid]
                    grid[row][col] = ""
                else:
                    old_uuid = grid[row][col]
                    if old_uuid and old_uuid in effects_props_group.keys():
                        sequencer_effect = effects_props_group[old_uuid]
                        sequencer_effect.attrs["effect_type"] = effect
                        if properties:
                            for key, value in properties.items():
                                sequencer_effect.attrs[f"prop_{key}"] = value
                    else:
                        new_uuid = create_sequencer_effect(effect, row, col, properties)
                        grid[row][col] = new_uuid
            
            set_effects_grid(grid)
            return jsonify({"effects": grid}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/em-effects/auto-set', methods=['POST'])
    def auto_set_em_effects():
        """Auto-set random EM effects for empty slots in EM zone"""
        try:
            grid = get_effects_grid()
            count = 0
            effect_types = ['AC', 'DC', 'AMF', 'CMF']
            
            with get_h5_file() as f:
                for row in range(2):
                    for col in range(8, 12):
                        if not grid[row][col]:
                            effect_type = random.choice(effect_types)
                            properties = {}
                            
                            if effect_type == 'AC':
                                properties = {
                                    'frequency': random.randint(50, 60),
                                    'voltage': random.randint(100, 240),
                                    'phase': random.randint(0, 360)
                                }
                            elif effect_type == 'DC':
                                properties = {
                                    'voltage': random.randint(5, 24),
                                    'current': random.uniform(0.1, 2.0)
                                }
                            elif effect_type == 'AMF':
                                properties = {
                                    'frequency': random.randint(1, 100),
                                    'amplitude': random.uniform(0.5, 5.0),
                                    'phase': random.randint(0, 360)
                                }
                            elif effect_type == 'CMF':
                                properties = {
                                    'strength': random.uniform(0.1, 1.0),
                                    'direction': random.choice(['N', 'S', 'E', 'W'])
                                }
                            
                            uuid = create_sequencer_effect(effect_type, row, col, properties)
                            grid[row][col] = uuid
                            count += 1
            
            set_effects_grid(grid)
            return jsonify({"success": True, "count": count}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/effects/<int:row>/<int:col>/properties', methods=['GET'])
    def get_effect_properties(row, col):
        """Get properties for a specific effect"""
        try:
            grid = get_effects_grid()
            uuid = grid[row][col]
            
            if not uuid:
                return jsonify({"properties": {}}), 200
            
            with get_h5_file() as f:
                effects_props_group = f.require_group("sequencer_effects_properties")
                if uuid in effects_props_group.keys():
                    sequencer_effect = effects_props_group[uuid]
                    effect_type = sequencer_effect.attrs.get("effect_type", "")
                    
                    properties = {}
                    for key in sequencer_effect.attrs.keys():
                        if key.startswith("prop_"):
                            prop_name = key[5:]
                            value = sequencer_effect.attrs[key]
                            if isinstance(value, bytes):
                                value = value.decode('utf-8')
                            properties[prop_name] = value
                    
                    return jsonify({
                        "effect_type": effect_type,
                        "properties": properties
                    }), 200
            
            return jsonify({"properties": {}}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500